# Generar archivo de ejemplo Bre-B

Crea `ejemplo_movimientos_breb.xlsx` con transferencias Bre-B **sintéticas**, en el mismo formato que espera `RutaDineroBreB.ipynb`. Sirve para probar el análisis sin datos reales.

El archivo tiene dos hojas:

- `movimientos`: una fila por transferencia que Cotrafa ve (el origen o el destino es una cuenta de Cotrafa).
- `cuentas`: fecha de apertura de las cuentas de Cotrafa.

Además se insertan cuatro tipologías de fraude conocidas para comprobar que el análisis las encuentra. La columna `TIPOLOGIA` solo existe en este ejemplo.

El repositorio incluye también `ejemplo_movimientos_breb.xls` (Excel 97-2003), que es el mismo archivo guardado en ese formato.

## 1. Parámetros

In [1]:
import numpy as np
import pandas as pd

SEMILLA = 2025
INICIO = pd.Timestamp("2026-11-02 00:00")  # lunes
DIAS = 7
N_COTRAFA = 400
N_EXTERNAS = 300
N_TRANSFERENCIAS = 6000
ARCHIVO_SALIDA = "ejemplo_movimientos_breb.xlsx"

ENTIDADES_EXTERNAS = ["ENTIDAD A", "ENTIDAD B", "ENTIDAD C", "ENTIDAD D", "ENTIDAD E"]
LLAVES = ["CELULAR", "DOCUMENTO", "CORREO", "ALFANUMERICA"]

rng = np.random.default_rng(SEMILLA)

## 2. Cuentas

400 cuentas de Cotrafa (`CTF-`) y 300 de otras entidades (`EXT-`). El 5% de las cuentas de Cotrafa son nuevas (menos de 60 días).

In [2]:
cuentas_ctf = [f"CTF-{i:04d}" for i in range(1, N_COTRAFA + 1)]
cuentas_ext = [f"EXT-{i:04d}" for i in range(1, N_EXTERNAS + 1)]

nueva = rng.random(N_COTRAFA) < 0.05
antiguedad = np.where(nueva, rng.integers(10, 60, N_COTRAFA), rng.integers(60, 5000, N_COTRAFA))
cuentas = pd.DataFrame({"cuenta": cuentas_ctf, "entidad": "COTRAFA", "antiguedad_dias": antiguedad})

entidad_ext = pd.DataFrame({"cuenta": cuentas_ext, "entidad": rng.choice(ENTIDADES_EXTERNAS, N_EXTERNAS)})
cuentas.head()

,cuenta,entidad,antiguedad_dias
0,CTF-0001,COTRAFA,114
1,CTF-0002,COTRAFA,448
2,CTF-0003,COTRAFA,1723
3,CTF-0004,COTRAFA,982
4,CTF-0005,COTRAFA,4678


## 3. Transferencias normales

Un 45% entra a Cotrafa desde otra entidad, un 33% sale de Cotrafa hacia otra entidad y un 22% va entre asociados. Los montos siguen una distribución log-normal con mediana cercana a $90.000 y las horas se concentran de día.

In [3]:
n = N_TRANSFERENCIAS
tipo = rng.choice(["sale", "interna", "entra"], n, p=[0.33, 0.22, 0.45])
origen = np.where(tipo == "entra", rng.choice(cuentas_ext, n), rng.choice(cuentas_ctf, n))
destino = np.where(tipo == "sale", rng.choice(cuentas_ext, n), rng.choice(cuentas_ctf, n))

dia = rng.integers(0, DIAS, n)
hora = np.clip(rng.normal(14, 4, n), 0, 23.99)
fecha = INICIO + pd.to_timedelta(dia, unit="D") + pd.to_timedelta(hora, unit="h")
monto = np.round(np.exp(rng.normal(np.log(90_000), 0.9, n)) / 100) * 100

normales = pd.DataFrame({"fecha": fecha, "origen": origen, "destino": destino, "monto": monto, "tipologia": "normal"})
normales = normales[normales.origen != normales.destino]
normales.describe()

,fecha,monto
count,5995,5.995000e+03
mean,2026-11-05 12:14:29.660425728,1.327059e+05
min,2026-11-02 00:00:00,3.900000e+03
25%,2026-11-03 15:36:59.035828224,4.950000e+04
50%,2026-11-05 13:02:54.854095104,8.790000e+04
75%,2026-11-07 09:48:45.609839616,1.604000e+05
max,2026-11-08 23:59:24,2.642600e+06
std,NaN,1.521041e+05


## 4. Tipologías de fraude

Cada tipología usa cuentas marcadas con `M` (`CTF-M01`). Primero se crean esas cuentas.

In [4]:
mulas = pd.DataFrame([
    ("CTF-M01", "COTRAFA", 12), ("CTF-M02", "COTRAFA", 25), ("CTF-M03", "COTRAFA", 8),
    ("CTF-M04", "COTRAFA", 40), ("CTF-M05", "COTRAFA", 18), ("CTF-M06", "COTRAFA", 33),
    ("CTF-M07", "COTRAFA", 20),
    ("EXT-M10", "ENTIDAD C", None), ("EXT-M11", "ENTIDAD B", None), ("EXT-M12", "ENTIDAD D", None),
    ("EXT-M13", "ENTIDAD E", None), ("EXT-M14", "ENTIDAD A", None), ("EXT-M15", "ENTIDAD C", None),
    ("EXT-M16", "ENTIDAD D", None), ("EXT-M17", "ENTIDAD B", None),
], columns=["cuenta", "entidad", "antiguedad_dias"])

antiguas = cuentas[cuentas.antiguedad_dias > 365].cuenta.to_numpy()
fraude = []  # aquí se van agregando las transferencias de cada tipología


def agregar(fecha, origen, destino, monto, tipologia):
    fraude.append({"fecha": fecha, "origen": origen, "destino": destino,
                   "monto": round(monto / 100) * 100, "tipologia": tipologia})


def minutos(m):
    return pd.Timedelta(minutes=int(m))

### 4.1 Estafa y estratificación

5 asociados engañados transfieren a una cuenta mula (`CTF-M01`). En minutos, la mula pasa el 60% a otra cuenta de Cotrafa (`CTF-M02`) y el 35% a otra entidad. `CTF-M02` lo saca a otra entidad, y de ahí se mueve a una tercera entidad (ese último paso Cotrafa no lo ve).

In [5]:
t0 = INICIO + pd.Timedelta(days=2, hours=10)
for victima in rng.choice(antiguas, 5, replace=False):
    t = t0 + minutos(rng.integers(0, 90))
    m = rng.uniform(1.5e6, 6e6)
    agregar(t, victima, "CTF-M01", m, "estafa_estratificacion")
    t1 = t + minutos(rng.integers(5, 25))
    agregar(t1, "CTF-M01", "CTF-M02", m * 0.60, "estafa_estratificacion")
    agregar(t1 + minutos(2), "CTF-M01", "EXT-M10", m * 0.35, "estafa_estratificacion")
    t2 = t1 + minutos(rng.integers(10, 40))
    agregar(t2, "CTF-M02", "EXT-M11", m * 0.60 * 0.95, "estafa_estratificacion")
    agregar(t2 + minutos(rng.integers(5, 30)), "EXT-M11", "EXT-M12", m * 0.60 * 0.9, "estafa_estratificacion")

### 4.2 Muchos a uno

14 cuentas de otras entidades envían montos pequeños a `CTF-M03` en 3 horas. La cuenta se vacía a la 1:30 a. m. hacia otra entidad.

In [6]:
total = 0
for origen_ in rng.choice(cuentas_ext, 14, replace=False):
    m = rng.uniform(150_000, 450_000)
    total += m
    agregar(INICIO + pd.Timedelta(days=4, hours=19) + minutos(rng.integers(0, 180)), origen_, "CTF-M03", m, "muchos_a_uno")
agregar(INICIO + pd.Timedelta(days=5, hours=1, minutes=30), "CTF-M03", "EXT-M13", total * 0.97, "muchos_a_uno")

### 4.3 Uno a muchos

Entran $9,8 millones a `CTF-M04` y se fraccionan en 11 transferencias menores a $1 millón. Una de ellas llega a otra mula (`CTF-M05`), que la saca de Cotrafa.

In [7]:
t = INICIO + pd.Timedelta(days=1, hours=15)
agregar(t, "EXT-M14", "CTF-M04", 9_800_000, "uno_a_muchos")
destinos = list(rng.choice(antiguas, 6, replace=False)) + ["CTF-M05"] + list(rng.choice(cuentas_ext, 4, replace=False))
for d in destinos:
    m = rng.uniform(700_000, 950_000)
    ts = t + minutos(rng.integers(5, 90))
    agregar(ts, "CTF-M04", d, m, "uno_a_muchos")
    if d == "CTF-M05":
        agregar(ts + minutos(rng.integers(10, 30)), "CTF-M05", "EXT-M16", m * 0.96, "uno_a_muchos")

### 4.4 Ciclo

El dinero entra a `CTF-M06`, pasa a `CTF-M07`, sale a otra entidad y vuelve a `CTF-M06` el mismo día.

In [8]:
t = INICIO + pd.Timedelta(days=3, hours=10)
agregar(t, "EXT-M17", "CTF-M06", 3_300_000, "ciclo")
agregar(t + minutos(60), "CTF-M06", "CTF-M07", 3_200_000, "ciclo")
agregar(t + minutos(100), "CTF-M07", "EXT-M15", 3_100_000, "ciclo")
agregar(t + minutos(280), "EXT-M15", "CTF-M06", 3_000_000, "ciclo")

pd.DataFrame(fraude).groupby("tipologia").monto.agg(["count", "sum"])

,count,sum
tipologia,,
ciclo,4,12600000
estafa_estratificacion,25,75640000
muchos_a_uno,15,7451900
uno_a_muchos,13,19694800


## 5. Unir y dejar solo lo que Cotrafa ve

Una transferencia entre dos cuentas de otras entidades no aparece en los registros de Cotrafa, así que no va en el archivo.

In [9]:
movs = pd.concat([normales, pd.DataFrame(fraude)], ignore_index=True).sort_values("fecha", ignore_index=True)

todas = pd.concat([cuentas[["cuenta", "entidad"]], entidad_ext, mulas[["cuenta", "entidad"]]])
entidad = todas.set_index("cuenta").entidad
movs["entidad_origen"] = movs.origen.map(entidad)
movs["entidad_destino"] = movs.destino.map(entidad)

visible = movs.entidad_origen.eq("COTRAFA") | movs.entidad_destino.eq("COTRAFA")
print(f"{(~visible).sum()} transferencias entre otras entidades quedan fuera del archivo")
movs = movs[visible].reset_index(drop=True)

5 transferencias entre otras entidades quedan fuera del archivo


## 6. Dar formato de extracto

Columnas con nombres de un extracto típico: fecha y hora separadas, valor en pesos y tipo de llave.

In [10]:
movimientos = pd.DataFrame({
    "ID_TRANSACCION": [f"BRB-{i:06d}" for i in range(1, len(movs) + 1)],
    "FECHA": movs.fecha.dt.normalize(),
    "HORA": movs.fecha.dt.strftime("%H:%M:%S"),
    "CUENTA_ORIGEN": movs.origen,
    "ENTIDAD_ORIGEN": movs.entidad_origen,
    "CUENTA_DESTINO": movs.destino,
    "ENTIDAD_DESTINO": movs.entidad_destino,
    "TIPO_LLAVE": rng.choice(LLAVES, len(movs), p=[0.5, 0.25, 0.1, 0.15]),
    "VALOR": movs.monto,
    "TIPOLOGIA": movs.tipologia,
})

aperturas = pd.concat([cuentas, mulas[mulas.entidad == "COTRAFA"]])
hoja_cuentas = pd.DataFrame({
    "CUENTA": aperturas.cuenta,
    "FECHA_APERTURA": (INICIO - pd.to_timedelta(aperturas.antiguedad_dias.astype(int), unit="D")).dt.normalize(),
})
movimientos.head()

,ID_TRANSACCION,FECHA,HORA,CUENTA_ORIGEN,ENTIDAD_ORIGEN,CUENTA_DESTINO,ENTIDAD_DESTINO,TIPO_LLAVE,VALOR,TIPOLOGIA
0,BRB-000001,2026-11-02,00:00:00,CTF-0377,COTRAFA,EXT-0043,ENTIDAD E,CELULAR,68500.0,normal
1,BRB-000002,2026-11-02,02:06:32,CTF-0177,COTRAFA,CTF-0118,COTRAFA,CELULAR,283600.0,normal
2,BRB-000003,2026-11-02,02:35:59,EXT-0217,ENTIDAD A,CTF-0317,COTRAFA,CORREO,123500.0,normal
3,BRB-000004,2026-11-02,02:44:18,CTF-0112,COTRAFA,EXT-0186,ENTIDAD B,ALFANUMERICA,174100.0,normal
4,BRB-000005,2026-11-02,03:09:41,EXT-0147,ENTIDAD B,CTF-0389,COTRAFA,DOCUMENTO,16900.0,normal


## 7. Guardar

In [11]:
with pd.ExcelWriter(ARCHIVO_SALIDA) as libro:
    movimientos.to_excel(libro, sheet_name="movimientos", index=False)
    hoja_cuentas.to_excel(libro, sheet_name="cuentas", index=False)
print(f"{ARCHIVO_SALIDA}: {len(movimientos):,} movimientos y {len(hoja_cuentas):,} cuentas")

ejemplo_movimientos_breb.xlsx: 6,047 movimientos y 407 cuentas
